#### Denoiser Pipeline

In [ ]:
class DenoiserModelHandler(ModelHandler[Tuple[str, AudioSamples], PredictionResult, Any]):
    def __init__(self, denoiser_model_path: str, device: str = "cuda"):
        self._denoiser_model_path = denoiser_model_path
        self._device = device
        self.denoiser = None

    def load_model(self) -> None:
        state_dict = torch.load(self._denoiser_model_path, map_location="cuda")
        self.denoiser = Denoiser()
        self.denoiser.load_state_dict(state_dict)
        self.denoiser.eval()
        self.denoiser.to(self._device)

    def run_inference(
        self,
        batch: Sequence[Tuple[str, AudioSamples]],
        model: Any,
        inference_args: Dict[str, Any] | None = None,
    ) -> Iterable[PredictionResult]:

        for key, audio_samples in batch:
            dwav = audio_samples.waveform
            sr = audio_samples.sample_rate
            # inference of denoising
            new_audio, _ = inference(model=model,dwav=dwav,sr=sr,device=self._device)
            yield PredictionResult(example=key, inference=new_audio)

#### Syncanet pipeline

In [ ]:
class SyncANetModelHandler(ModelHandler[Tuple[str, AudioSamples], PredictionResult, Any]):
    def __init__(self, sync_anet_model_path: str, device: str = "cuda"):
        self._sync_anet_model_path = sync_anet_model_path
        self._device = device
        self.sync_anet_model = None

    def load_model(self) -> Any:
        state_dict = torch.load(self._sync_anet_model_path, map_location="cpu")
        self.sync_anet_model = SyncANet()
        self.sync_anet_model.load_state_dict(state_dict)
        self.sync_anet_model.eval()
        self.sync_anet_model.to(self._device)

    def run_inference(
        self,
        batch: Sequence[Tuple[str, AudioSamples]],
        model: Any,
        inference_args: Dict[str, Any] | None = None,
    ) -> Iterable[PredictionResult]:
        
        for key, audio_samples in batch:
            audio_data = audio_samples.waveform
            # infernece for syncanet
            out = decode_one_audio_mossformergan_se_16k(model=model,device=self._device,inputs=audio_data)
            yield PredictionResult(example=key, inference=out)

#### custom run sudio inference

In [ ]:
class EnhanceRunAudioInference(beam.PTransform):
    def __init__(self, SyncANetModelHandler: ModelHandler, DenoiserModelHandler: ModelHandler):
        self.SyncANetModelHandler = SyncANetModelHandler
        self.DenoiserModelHandler = DenoiserModelHandler

    def expand(self, pcoll: beam.PCollection) -> beam.PCollection:
        enhanced = (
            pcoll
            | "denoiser_enhanced" >> RunInference(self.DenoiserModelHandler),
            | "syncanet_enhanced" >> RunInference(self.SyncANetModelHandler)
        )
        return syncanet_enhanced

#### example pipeline

In [ ]:
with beam.Pipeline(InteractiveRunner(), options=PipelineOptions()) as p:
    output = (
        p
        | "EnhanceAudio" >> EnhanceRunAudioInference(SyncANetModelHandler=syncanet_mh,DenoiserModelHandler=DenoiserModelHandler)
    )